<a href="https://colab.research.google.com/github/milaho1006/ferrero-rocher-sales-analytics/blob/main/Ferrero_Data_Preparation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Unwrap the Insights - Data preparation
Independent BI portfolio project using the public Ferrero Rocher sales dataset by Arannayava Debnath on Kaggle.

**Purpose:** prepare trustworthy reporting data for Tableau and subsequent SQL analysis. This is not an official Ferrero report; real-world provenance and currency are unverified.

## How to run
In Jupyter, place the original `ferrero_rocher_sales_dataset.csv` beside this notebook (or in `../upload/`), then choose **Run All**. In Google Colab, upload the CSV to the session before running. Dependencies: pandas and numpy. Outputs are written to `outputs/`.

## Business brief
Audience: a hypothetical Revenue & Trade Management team.
1. How do product volume and recorded revenue differ?
2. How do recorded sales patterns vary across promotion groups?
3. What decisions or further tests can this evidence support?

One row is treated as one supplied sales record. There is no transaction/order identifier, so row counts must not be presented as distinct orders or customers.

In [1]:
!git clone https://github.com/milaho1006/ferrero-rocher-sales-analytics.git

Cloning into 'ferrero-rocher-sales-analytics'...
remote: Enumerating objects: 32, done.
remote: Counting objects: 100% (32/32), done.
remote: Compressing objects: 100% (26/26), done.
remote: Total 32 (delta 8), reused 0 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (32/32), 364.62 KiB | 2.51 MiB/s, done.
Resolving deltas: 100% (8/8), done.


In [2]:
import pandas as pd

df = pd.read_csv(
    "/content/ferrero-rocher-sales-analytics/data/raw/ferrero_rocher_sales_dataset.csv"
)

df.head()

,Store,Country,SKU,Date,Channel,Promotion,Units Sold,Unit Price,Discount,Revenue,Margin %,Margin
0,Tesco,UAE,Ferrero Rocher T3,45911,Wholesale,Diwali Promo,153,120,0.15,15606.0,30,4681.80
1,Tesco,USA,Ferrero Rocher 32pc,45940,Wholesale,Diwali Promo,143,750,0.15,91162.5,34,30995.25
2,Carrefour,Canada,Ferrero Rocher 32pc,45733,Retail,Buy1Get1,37,750,0.50,13875.0,37,5133.75
3,Walmart,India,Ferrero Collection 15pc,45945,Wholesale,NaN,101,650,0.00,65650.0,40,26260.00
4,Tesco,UK,Ferrero Rocher 16pc,45814,Retail,NaN,166,350,0.00,58100.0,30,17430.00


In [3]:
from pathlib import Path
import pandas as pd
import numpy as np

candidates = [Path('ferrero_rocher_sales_dataset.csv'), Path('../upload/ferrero_rocher_sales_dataset.csv'), Path('/content/ferrero_rocher_sales_dataset.csv')]
source = next((p for p in candidates if p.exists()), None)
if source is None:
    raise FileNotFoundError('Upload the original sales CSV beside the notebook, then run again.')
raw = pd.read_csv(source)
out = Path('outputs')
out.mkdir(exist_ok=True)
print(f'Loaded {len(raw):,} records and {len(raw.columns)} columns.')
raw.head()

Loaded 20,000 records and 12 columns.


,Store,Country,SKU,Date,Channel,Promotion,Units Sold,Unit Price,Discount,Revenue,Margin %,Margin
0,Tesco,UAE,Ferrero Rocher T3,45911,Wholesale,Diwali Promo,153,120,0.15,15606.0,30,4681.80
1,Tesco,USA,Ferrero Rocher 32pc,45940,Wholesale,Diwali Promo,143,750,0.15,91162.5,34,30995.25
2,Carrefour,Canada,Ferrero Rocher 32pc,45733,Retail,Buy1Get1,37,750,0.50,13875.0,37,5133.75
3,Walmart,India,Ferrero Collection 15pc,45945,Wholesale,NaN,101,650,0.00,65650.0,40,26260.00
4,Tesco,UK,Ferrero Rocher 16pc,45814,Retail,NaN,166,350,0.00,58100.0,30,17430.00


## 1. Profile before changing anything
Preserve the original input. Missing Promotion means no promotion according to the supplied dictionary. Other missing values require investigation. Exact duplicates are flagged, not automatically removed: identical records can represent separate sales.

In [4]:
expected = ['Store', 'Country', 'SKU', 'Date', 'Channel', 'Promotion', 'Units Sold', 'Unit Price', 'Discount', 'Revenue', 'Margin %', 'Margin']
assert set(raw.columns) == set(expected), 'Unexpected input schema.'
profile = pd.DataFrame({'type': raw.dtypes.astype(str), 'missing': raw.isna().sum(), 'distinct_non_null': raw.nunique()})
print('Exact duplicate rows:', raw.duplicated().sum())
profile

Exact duplicate rows: 0


,type,missing,distinct_non_null
Store,object,0,28
Country,object,0,7
SKU,object,0,5
Date,int64,0,365
Channel,object,0,4
Promotion,object,3256,5
Units Sold,int64,0,190
Unit Price,int64,0,5
Discount,float64,0,6
Revenue,float64,0,3644


## 2. Clean labels and dates
Trim leading/trailing spaces. Do not silently rename potentially misspelled retailers without evidence. Convert Excel serial dates using origin 1899-12-30. Add a source row number for traceability; this is not a business transaction ID. Preserve channel labels: Online/Offline and Retail/Wholesale may describe overlapping concepts and need business clarification.

In [5]:
clean = raw.copy()
text_cols = ['Store', 'Country', 'SKU', 'Channel', 'Promotion']
trim_counts = {}
for col in text_cols:
    before = clean[col].astype('string')
    after = before.str.strip()
    trim_counts[col] = int((before != after).fillna(False).sum())
    clean[col] = after
no_promo_count = int(clean['Promotion'].isna().sum())
clean['Promotion'] = clean['Promotion'].fillna('No promotion')
clean['Date'] = pd.to_datetime(clean['Date'], unit='D', origin='1899-12-30', errors='coerce')
clean.insert(0, 'Source Row Number', np.arange(2, len(clean) + 2))
clean['Year'] = clean['Date'].dt.year
clean['Month Number'] = clean['Date'].dt.month
clean['Month Start'] = clean['Date'].dt.to_period('M').dt.to_timestamp()
clean['Quarter'] = 'Q' + clean['Date'].dt.quarter.astype(str)
print('Whitespace changes:', trim_counts)
print('No-promotion labels:', no_promo_count)
print('Date coverage:', clean['Date'].min(), 'to', clean['Date'].max())
clean.head()

Whitespace changes: {'Store': 1363, 'Country': 0, 'SKU': 0, 'Channel': 0, 'Promotion': 0}
No-promotion labels: 3256
Date coverage: 2025-01-01 00:00:00 to 2025-12-31 00:00:00


,Source Row Number,Store,Country,SKU,Date,Channel,Promotion,Units Sold,Unit Price,Discount,Revenue,Margin %,Margin,Year,Month Number,Month Start,Quarter
0,2,Tesco,UAE,Ferrero Rocher T3,2025-09-11,Wholesale,Diwali Promo,153,120,0.15,15606.0,30,4681.80,2025,9,2025-09-01,Q3
1,3,Tesco,USA,Ferrero Rocher 32pc,2025-10-10,Wholesale,Diwali Promo,143,750,0.15,91162.5,34,30995.25,2025,10,2025-10-01,Q4
2,4,Carrefour,Canada,Ferrero Rocher 32pc,2025-03-17,Retail,Buy1Get1,37,750,0.50,13875.0,37,5133.75,2025,3,2025-03-01,Q1
3,5,Walmart,India,Ferrero Collection 15pc,2025-10-15,Wholesale,No promotion,101,650,0.00,65650.0,40,26260.00,2025,10,2025-10-01,Q4
4,6,Tesco,UK,Ferrero Rocher 16pc,2025-06-06,Retail,No promotion,166,350,0.00,58100.0,30,17430.00,2025,6,2025-06-01,Q2


## 3. Validate before export
Use small numeric tolerances for floating-point comparisons. Revenue should equal units × price × (1 − discount); supplied margin should equal revenue × margin percentage / 100. These checks confirm internal consistency, not business authenticity.

In [6]:
checks = {
    'Rows retained': len(clean) == len(raw),
    'No missing output values': not clean.isna().any().any(),
    'Dates in documented 2025 range': clean['Date'].between('2025-01-01', '2025-12-31').all(),
    'Positive integer units': ((clean['Units Sold'] > 0) & (clean['Units Sold'] % 1 == 0)).all(),
    'Positive prices': (clean['Unit Price'] > 0).all(),
    'Discounts between zero and one': clean['Discount'].between(0, 1).all(),
    'Margin percentage within zero to 100': clean['Margin %'].between(0, 100).all(),
    'Revenue formula reconciles': np.isclose(clean['Revenue'], clean['Units Sold'] * clean['Unit Price'] * (1-clean['Discount']), rtol=1e-9, atol=0.01).all(),
    'Margin formula reconciles': np.isclose(clean['Margin'], clean['Revenue'] * clean['Margin %']/100, rtol=1e-9, atol=0.01).all(),
    'Revenue total preserved': np.isclose(raw['Revenue'].sum(), clean['Revenue'].sum()),
    'Margin total preserved': np.isclose(raw['Margin'].sum(), clean['Margin'].sum())
}
quality = pd.DataFrame([{'Check': k, 'Passed': bool(v)} for k,v in checks.items()])
print(quality.to_string(index=False))
assert quality['Passed'].all(), 'Investigate failed checks before using this dataset.'

                               Check  Passed
                       Rows retained    True
            No missing output values    True
      Dates in documented 2025 range    True
              Positive integer units    True
                     Positive prices    True
      Discounts between zero and one    True
Margin percentage within zero to 100    True
          Revenue formula reconciles    True
           Margin formula reconciles    True
             Revenue total preserved    True
              Margin total preserved    True


## 4. Reporting definitions and limitations
- **Units sold:** sum of supplied Units Sold; pack sizes differ, so this is not individual chocolates sold.
- **Recorded revenue:** sum of Revenue within a selected country. Currency is unspecified even within countries; do not label as AUD or rank country revenues as economically comparable. Any all-country aggregate is only a dataset arithmetic total.
- **Supplied margin amount:** sum of Margin. No cost basis is provided, so do not relabel it as independently verified gross or net profit.
- **Weighted margin rate:** SUM(Margin) / SUM(Revenue). Format as a percentage; do not average row margin percentages.
- **Units per record:** SUM(Units Sold) / COUNT(records). It is not average basket size.
- **Weighted discount:** SUM(Units × Price × Discount) / SUM(Units × Price), a share of recorded pre-discount value. This is not promotional ROI or incremental loss.

Promotion comparisons are observational. Each promotion is tied to a fixed discount; these effects cannot be separated. Differences in product mix, record counts and channels can influence totals. There is no control group, promotion spend, customer identifier or inventory history. Do not infer causality, elasticity, loyalty, ROI or stockout effects. One year is insufficient to establish recurring seasonal behaviour.

In [7]:
promotion_structure = pd.crosstab(clean['Promotion'], clean['Discount'])
print(promotion_structure.to_string())
print('\nRecords per country:')
print(clean.groupby('Country').size().to_string())
print('\nChannel counts (definitions require clarification):')
print(clean['Channel'].value_counts().to_string())

Discount         0.00  0.10  0.15  0.20  0.25  0.50
Promotion                                          
Buy1Get1            0     0     0     0     0  3321
Christmas Promo     0     0     0  3277     0     0
Diwali Promo        0     0  3492     0     0     0
NewYear Sale        0     0     0     0  3322     0
No promotion     3256     0     0     0     0     0
Valentine Promo     0  3332     0     0     0     0

Records per country:
Country
Australia    2777
Canada       2958
India        2740
Singapore    2846
UAE          2818
UK           3010
USA          2851

Channel counts (definitions require clarification):
Channel
Online       7284
Wholesale    6326
Retail       6184
Offline       206


## 5. Export for Tableau
The clean CSV retains the supplied measures plus reporting dates and lineage. It does not overwrite the original. Import Date and Month Start as dates in Tableau; Discount is a decimal fraction, but Margin % is a 25–40 numeric percentage. Use calculated weighted margin for aggregated views. Start the dashboard with a single-country filter, for example Australia, and retain the currency caveat.

In [8]:
clean.to_csv(out / 'ferrero_sales_clean.csv', index=False, date_format='%Y-%m-%d')
quality.to_csv(out / 'data_quality_checks.csv', index=False)
changes = pd.DataFrame([
    {'Change': 'Trim retailer whitespace', 'Affected rows': trim_counts['Store']},
    {'Change': 'Label null promotion as No promotion', 'Affected rows': no_promo_count},
    {'Change': 'Convert Excel dates', 'Affected rows': len(clean)},
    {'Change': 'Rows deleted', 'Affected rows': 0}
])
changes.to_csv(out / 'cleaning_log.csv', index=False)
reloaded = pd.read_csv(out / 'ferrero_sales_clean.csv')
assert len(reloaded) == len(raw)
assert np.isclose(reloaded['Revenue'].sum(), raw['Revenue'].sum())
print('Export and read-back checks passed. Files:', [p.name for p in out.glob('*.csv')])

Export and read-back checks passed. Files: ['ferrero_sales_clean.csv', 'cleaning_log.csv', 'data_quality_checks.csv']


## Next step
- Define the reporting model and implement SQL queries. Reconcile those query totals against this clean dataset before building Tableau.
- Suggested dimensions: date, product, country, retailer and channel, with promotion/discount relationships documented.
- Do not assume a retailer name identifies a physical store location.